# TVM Prep Guide

This is the main guide for the repository. The markdown files explain how to open the devcontainer and install dependencies; this notebook explains how to use the project once the container is ready.

Run this notebook from top to bottom after these setup commands have completed in the repository root:

```bash
bash .devcontainer/scripts/install-python-deps.sh requirements.txt
bash .devcontainer/scripts/build-tvm.sh
```

The notebook starts with a tiny offline PyTorch model so the first pass is fast and deterministic. After that, it explains how the same helpers are used for larger models, other frontends, target profiles, Python runtime validation, and the C++ runner.

## What TVM Is Doing Here

Apache TVM is a compiler stack for machine-learning models. In this repository the workflow has five stages:

1. Start from a model in a framework such as PyTorch, TensorFlow/Keras, ONNX, or TFLite.
2. Convert that model into TVM Relay, which is TVM's high-level graph representation.
3. Compile Relay for a target profile, such as local x86_64 Linux or Raspberry Pi AArch64.
4. Export graph-executor artifacts: graph JSON, parameter bytes, compiled library, and metadata.
5. Load those artifacts in a runtime and run inference.

The notebook uses `compilation/compile.py` for the compile/export step and `examples/python/` for runtime execution. The code stays close to the TVM APIs so it is clear what TVM is doing.

In [8]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent

EXAMPLES    = ROOT / 'examples' / 'python'
COMPILATION = ROOT / 'compilation'
for p in (EXAMPLES, COMPILATION):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

print('Python:', sys.version.split()[0])
print('Repository:', ROOT)


Python: 3.11.15
Repository: /workspaces/TVM-Prep-guide


## Validate The Environment

A TVM setup can fail in several places: the Python frameworks may be missing, TVM may not find its compiled shared libraries, or the cross-compilers may not exist. This cell checks those pieces before any model compilation begins.

The imports run in subprocesses so noisy framework startup logs, especially TensorFlow CUDA messages, do not make the notebook look failed when the import actually succeeded.

In [9]:
import shutil
import subprocess

from targets import TARGETS

checks = {
    'TVM':          'import tvm; print(tvm.__version__)',
    'PyTorch':      'import torch; print(torch.__version__)',
    'TorchVision':  'import torchvision; print(torchvision.__version__)',
    'TensorFlow':   'import tensorflow as tf; print(tf.__version__)',
    'ONNX':         'import onnx; print(onnx.__version__)',
    'ONNX Runtime': 'import onnxruntime; print(onnxruntime.__version__)',
}

for label, code in checks.items():
    result = subprocess.run([sys.executable, '-c', code], text=True, capture_output=True)
    if result.returncode != 0:
        raise RuntimeError(f'{label} import failed:\n{result.stderr}')
    print(f'{label}:', result.stdout.strip().splitlines()[-1])

print('\nToolchain:')
for tool in ['gcc', 'g++', 'aarch64-linux-gnu-gcc', 'arm-linux-gnueabihf-gcc', 'emcc', 'cmake', 'ninja']:
    print(f'{tool}:', shutil.which(tool))


TVM: 0.19.0
PyTorch: 2.3.1+cu121
TorchVision: 0.18.1+cu121
TensorFlow: 2.16.2
ONNX: 1.16.2
ONNX Runtime: 1.18.1

Toolchain:
gcc: /usr/bin/gcc
g++: /usr/bin/g++
aarch64-linux-gnu-gcc: /usr/bin/aarch64-linux-gnu-gcc
arm-linux-gnueabihf-gcc: /usr/bin/arm-linux-gnueabihf-gcc
emcc: /usr/bin/emcc
cmake: /usr/bin/cmake
ninja: /usr/bin/ninja


## Pick A Target Profile

A TVM target describes the machine code TVM should generate. The same model can be compiled for local Linux, Raspberry Pi, WebAssembly, Vulkan, or C source export by changing the profile.

For this notebook, use `x86_64` because it runs inside the devcontainer. Cross targets are useful after local validation works.

In [10]:
for name, profile in sorted(TARGETS.items()):
    print(f'{name}')
    print(f'  target: {profile["target"]}')
    if 'host' in profile:
        print(f'  host:   {profile["host"]}')
    print(f'  output: model.{profile["ext"]}')
    if 'cc' in profile:
        print(f'  cc:     {profile["cc"]}')
    print()

TARGET_PROFILE = 'x86_64'


c
  target: c
  output: model.tar

native
  target: llvm -mcpu=native
  output: model.so
  cc:     gcc

raspi4_aarch64
  target: llvm -mtriple=aarch64-linux-gnu -mcpu=cortex-a72 -mattr=+neon
  host:   llvm -mtriple=aarch64-linux-gnu
  output: model.so
  cc:     aarch64-linux-gnu-gcc

raspi_armv7
  target: llvm -mtriple=armv7-linux-gnueabihf -mcpu=cortex-a72 -mattr=+neon,+vfp3
  host:   llvm -mtriple=armv7-linux-gnueabihf
  output: model.so
  cc:     arm-linux-gnueabihf-gcc

vulkan
  target: vulkan
  host:   llvm -mtriple=x86_64-linux-gnu
  output: model.so

wasm32
  target: llvm -mtriple=wasm32-unknown-unknown-wasm
  output: model.wasm
  cc:     emcc

x86_64
  target: llvm -mtriple=x86_64-linux-gnu -mcpu=x86-64
  output: model.so
  cc:     gcc



## Compile A Small Model

The first compile uses a tiny in-notebook PyTorch model so the notebook does not need network access or downloaded model weights. A pretrained ResNet compile is useful, but it requires downloading weights and takes longer.

This small classifier still exercises the important TVM steps: tracing a PyTorch module, converting it to Relay, compiling it, exporting artifacts, and running those artifacts.


In [11]:
import torch
from tvm import relay


class TinyImageClassifier(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.features = torch.nn.Sequential(
            torch.nn.Conv2d(3, 8, kernel_size=3, padding=1),
            torch.nn.ReLU(),
            torch.nn.AdaptiveAvgPool2d((1, 1)),
        )
        self.classifier = torch.nn.Linear(8, 4)

    def forward(self, x):
        x = self.features(x)
        x = torch.flatten(x, 1)
        return self.classifier(x)


torch.manual_seed(0)
torch.set_grad_enabled(False)

MODEL_NAME  = 'tiny_image_classifier'
INPUT_NAME  = 'input0'
INPUT_SHAPE = (1, 3, 32, 32)
model = TinyImageClassifier().eval()

print('Model :', MODEL_NAME)
print('Input :', INPUT_NAME, INPUT_SHAPE)


Model : tiny_image_classifier
Input : input0 (1, 3, 32, 32)


## Import The Model Into Relay

Relay is TVM's graph IR. For PyTorch, we trace the module to TorchScript with a zero-valued example tensor, then call `relay.frontend.from_pytorch`. The output is an `IRModule` plus a parameter dict.

At this stage TVM has understood the model graph but has not generated target-specific machine code yet.


In [12]:
example = torch.zeros(INPUT_SHAPE, dtype=torch.float32)
scripted = torch.jit.trace(model, example).eval()

mod, params = relay.frontend.from_pytorch(scripted, [(INPUT_NAME, INPUT_SHAPE)])
print(type(mod).__name__)
print('Parameter tensors:', len(params))


IRModule
Parameter tensors: 4


## Compile And Export Artifacts

`build_and_save` from `compilation/compile.py` compiles the Relay module with `relay.build` and writes a portable artifact directory:

- `model.json`: graph-executor graph
- `model.params`: serialized parameter dictionary
- `model.so`, `model.tar`, or `model.wasm`: compiled target library
- `metadata.json`: model name, input name, input shape, target, and library filename

Generated outputs go under `examples/artifacts/`, which is ignored by git except for `.gitkeep`.


In [13]:
import json
from compile import build_and_save

artifact_dir = build_and_save(
    mod, params,
    model_name=MODEL_NAME,
    target_name=TARGET_PROFILE,
    output_dir=ROOT / 'examples' / 'artifacts',
    input_name=INPUT_NAME,
    input_shape=INPUT_SHAPE,
)
print('Artifacts:', artifact_dir.relative_to(ROOT))
for path in sorted(artifact_dir.iterdir()):
    print('-', path.name)

metadata = json.loads((artifact_dir / 'metadata.json').read_text())
metadata


Artifacts: examples/artifacts/tiny_image_classifier/x86_64
- metadata.json
- model.json
- model.params
- model.so


{'model': 'tiny_image_classifier',
 'input_name': 'input0',
 'input_shape': [1, 3, 32, 32],
 'input_dtype': 'float32',
 'target': 'x86_64',
 'target_str': 'llvm -mtriple=x86_64-linux-gnu -mcpu=x86-64',
 'library': 'model.so'}

## Run The Exported Artifacts With Python

The Python runtime path mirrors what a deployment program does:

1. Read `metadata.json` to learn the input name and compiled library file.
2. Load `model.json`, `model.params`, and the compiled library.
3. Create a TVM graph executor module.
4. Set the model input, run inference, and read output tensor 0.

The input below is deterministic synthetic data. For ImageNet models, the CLI uses image preprocessing from `tvm_prep.preprocess` instead.

In [14]:
import numpy as np

from tvm_prep.runtime import run_graph_executor, topk

input_data = np.linspace(-1.0, 1.0, num=np.prod(INPUT_SHAPE), dtype='float32').reshape(INPUT_SHAPE)
output = run_graph_executor(artifact_dir, input_data)

print('output shape:', output.shape)
for rank, (idx, prob, label) in enumerate(topk(output, k=4), start=1):
    print(f'{rank}: id={idx} p={prob:.6f} {label}')


output shape: (1, 4)
1: id=1 p=0.393344 <no-label>
2: id=2 p=0.261390 <no-label>
3: id=0 p=0.183968 <no-label>
4: id=3 p=0.161298 <no-label>


## Compile Real Models With The CLI

Once the small model works, use the CLI for real models. The CLI in `compilation/compile.py` calls the same `build_and_save` function used above.

PyTorch torchvision example:

```bash
python compilation/compile.py \
  --frontend pytorch \
  --model resnet18 \
  --target x86_64

python examples/python/run_model.py \
  --artifact-dir examples/artifacts/resnet18/x86_64 \
  --image examples/assets/cat.png
```

Supported starter PyTorch models are `resnet18`, `mobilenet_v2`, `squeezenet1_1`, and `shufflenet_v2_x1_0`. TensorFlow starter models are `mobilenet_v2` and `resnet50`.

ONNX and TFLite need a file path, input name, and input shape because those details are model-specific:

```bash
python compilation/compile.py \
  --frontend onnx \
  --model-path path/to/model.onnx \
  --input-name input0 \
  --input-shape 1,3,224,224 \
  --target x86_64
```

## Compile For Other Targets

Change only `--target` when moving from local validation to another supported profile:

```bash
python compilation/compile.py --frontend pytorch --model resnet18 --target raspi4_aarch64
python compilation/compile.py --frontend pytorch --model resnet18 --target raspi_armv7
python compilation/compile.py --frontend pytorch --model resnet18 --target wasm32
python compilation/compile.py --frontend pytorch --model resnet18 --target c
```

Cross-compilation is only half the job. The target device also needs a compatible TVM runtime library. Build it with:

```bash
bash compilation/build_runtime.sh raspi4_aarch64
# Output: examples/artifacts/runtime/raspi4_aarch64/libtvm_runtime.so
```

Copy that file alongside the compiled model artifacts to the target device.

## Use The C++ Runtime Example

The C++ runner consumes the same artifact directory as the Python runtime. Build it after compiling a model:

```bash
cmake -S examples/cpp -B examples/cpp/build -G Ninja
cmake --build examples/cpp/build

examples/cpp/build/run_tvm_graph \
  --artifact-dir examples/artifacts/resnet18/x86_64 \
  --image examples/assets/cat.png
```

For cross-compiled artifacts, the C++ runner, compiled model library, TVM runtime library, and target OS ABI must all match.

## Where To Change Things

- Add target profiles in `compilation/targets.py`.
- Change compile/export behavior in `compilation/compile.py`.
- Change Python runtime behavior in `examples/python/tvm_prep/runtime.py`.
- Change image preprocessing in `examples/python/tvm_prep/preprocess.py`.

Keep generated model outputs under `examples/artifacts/`. Do not commit generated `.so`, `.params`, `.json`, ONNX, or artifact directories.
